# E03 Wine Quality：回归 / 有序分类与不均衡

> Issue #9 / Epic #6 / 路线图 E03

## 你将学到
1. `quality` 是有序整数：可当回归，也可当多分类
2. 极端分数（3、9）样本极少 → Accuracy 会骗人
3. 红/白酒化学特征不同，是否分开建模要对比

## 数据
- 优先：`data/winequality-red.csv`（UCI，`;` 分隔）
- 回退：OpenML `wine-quality-red`（需网络）
- **注意**：不是 `sklearn.datasets.load_wine`


In [ ]:
QUICK = True
RANDOM_STATE = 42

# 一键就绪：关警告 + 打印环境 + 设种子（勿再叠 JE.no_warning / filterwarnings）
from wanglaoshi import ready
ready(seed=RANDOM_STATE)


## 1. 加载 Wine Quality（带本地/OpenML 回退）


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

path = Path('../../data/winequality-red.csv')
df = None
if path.exists():
    df = pd.read_csv(path, sep=';')
    print('loaded local', path, df.shape)
else:
    try:
        from sklearn.datasets import fetch_openml
        raw = fetch_openml('wine-quality-red', version=1, as_frame=True, parser='auto')
        df = raw.frame.copy()
        # OpenML 列名可能是 class / quality
        if 'quality' not in df.columns and 'class' in df.columns:
            df = df.rename(columns={'class': 'quality'})
        print('loaded OpenML wine-quality-red', df.shape)
    except Exception as e:
        print('无法加载数据:', e)
        print('请下载 https://archive.ics.uci.edu/dataset/186/wine+quality 到 data/winequality-red.csv')

if df is not None:
    df['quality'] = pd.to_numeric(df['quality'], errors='coerce')
    print(df['quality'].value_counts().sort_index())
    display(df.head())


## 2. EDA：目标不均衡与特征相关


In [ ]:
from wanglaoshi import DataAnalyzer

assert df is not None, '请先成功加载 df'
eda_df = df.sample(min(len(df), 5000), random_state=RANDOM_STATE) if QUICK else df
analyzer = DataAnalyzer(eda_df)
print(analyzer.get_basic_info())
display(analyzer.missing_value_analysis())
display(analyzer.basic_statistics().head(20))
print('quality distribution %:\n', (df['quality'].value_counts(normalize=True).sort_index() * 100).round(1))


## 3. 双视角：回归 vs 有序分类划分


In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = [c for c in df.columns if c != 'quality']
X = df[feature_cols].apply(pd.to_numeric, errors='coerce')
y = df['quality'].astype(int)

# 分层按 quality（分类视角）
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
med = Xtr.median()
Xtr_f, Xte_f = Xtr.fillna(med), Xte.fillna(med)
print(Xtr_f.shape, 'train quality counts', ytr.value_counts().sort_index().to_dict())


## 4. 回归基线（把 quality 当连续）


In [ ]:
from wanglaoshi import CompetitionTools as CT
from sklearn.metrics import mean_absolute_error, mean_squared_error

baseline = CT.QuickBaseline()
reg_results = baseline.run_all_models(Xtr_f, ytr, Xte_f, yte, task_type='regression')
reg_cmp = baseline.compare_models(reg_results, top_n=5)
display(reg_cmp)

best_reg = reg_cmp.iloc[0]['Model']
pred_reg = reg_results[best_reg]['predictions']
print('MAE', mean_absolute_error(yte, pred_reg))
print('RMSE', mean_squared_error(yte, pred_reg, squared=False))
# 有序性检查：把回归预测四舍五入后看 Accuracy
pred_round = np.clip(np.rint(pred_reg), y.min(), y.max()).astype(int)
print('rounded accuracy', (pred_round == yte.to_numpy()).mean())


## 5. 分类基线（注意极端类稀少）


In [ ]:
from sklearn.metrics import classification_report, f1_score

clf_results = baseline.run_all_models(Xtr_f, ytr, Xte_f, yte, task_type='classification')
clf_cmp = baseline.compare_models(clf_results, top_n=5)
display(clf_cmp)
best_clf = clf_cmp.iloc[0]['Model']
pred_clf = clf_results[best_clf]['predictions']
print(classification_report(yte, pred_clf, zero_division=0))
print('macro-F1', f1_score(yte, pred_clf, average='macro', zero_division=0))


## 6. 模拟提交与局限


In [ ]:
from pathlib import Path
from wanglaoshi import CompetitionTools as CT

out = Path('outputs/03_wine_quality'); out.mkdir(parents=True, exist_ok=True)
sample = pd.DataFrame({'Id': np.arange(len(pred_reg)), 'quality': yte.to_numpy()})
sample.to_csv(out / 'sample_submission.csv', index=False)
gen = CT.SubmissionGenerator()
sub = gen.create_submission(
    pred_reg, str(out / 'sample_submission.csv'), str(out / 'submission.csv'), prediction_type='label'
)
print(gen.validate_submission(str(out / 'submission.csv'), str(out / 'sample_submission.csv')))


## 7. 检查清单
- [x] 区分回归 / 分类两种建模视角
- [x] 报告 macro-F1（不只 Accuracy）
- [x] 本地/OpenML 双路径加载
- [ ] 红白酒合并 vs 分开建模对比
- [ ] 用 ordinal 模型或相邻类惩罚
